In [1]:
# Mini Project 1 : Building a Question-Answering System with LlamaIndex

  # Load documents from a local directory using SimpleDirectoryReader
  # Set up and configure HuggingFace LLM and embedding models
  # Create a VectorStoreIndex with llama_index
  # Persist and reload indexes from disk
  # Query the index and retrieve well-structured outputs

# This repository contains a minimal example of building a Question-Answering (QA) system using [LlamaIndex](https://github.com/run-llama/llama_index).

## Requirements
  # Python 3.6+
  # llama-index`
  # transformers`
  # sentence-transformers`

In [2]:
## Installing Dependencies with:
  # bash
  # pip install llama-index transformers sentence-transformers

# Install the dependencies with:
%pip install llama-index transformers sentence-transformers
%pip install llama-index-llms-huggingface llama-index-embeddings-huggingface
!python qa_system.py
%pip install PyPDF2 pdf2image pytesseract pillow

python3: can't open file '/content/qa_system.py': [Errno 2] No such file or directory


Importing Modules
The script imports the Path utilities from pathlib, key classes from llama_index (directory reader, vector index, LLM, and embedding model), as well as the os and requests modules.
These libraries respectively handle file paths, process text data, and download a file.

In [19]:
# Importing Modules, in particular :
  # the Path utilities from pathlib,
  # key classes from llama_index (directory reader, vector index, LLM, and embedding model)
  # os and requests modules.

  # These libraries respectively handle file paths, process text data, and download a file.

from pathlib import Path

from llama_index.core.readers import SimpleDirectoryReader
from llama_index.core import VectorStoreIndex, ServiceContext, StorageContext, load_index_from_storage
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings

In [20]:
# Preparing the Data Directory
    # os.makedirs('data') is executed if the data folder does not exist, to store the source documents (in this case, a downloaded PDF).

import os
if not os.path.exists('data'):
    os.makedirs('data')

In [16]:
# PDF_PATH = r"C:\Users\cibei\Desktop\Introduction to Deep Learning.pdf"

In [17]:
"""Simple Question-Answering system using LlamaIndex.

This script demonstrates how to load documents from a local directory,
create a vector index using HuggingFace models, persist it to disk,
reload it, and query for answers.
"""
# Global Constants
    # DATA_DIR: location where documents to be indexed will be stored.
    # PERSIST_DIR: location where the vector index will be saved to disk.

DATA_DIR = Path("data")
PERSIST_DIR = Path("storage")

In [21]:
# Building the Index (build_index)
    # SimpleDirectoryReader(DATA_DIR).load_data() loads all files in the data folder.
    # Settings.llm and Settings.embed_model configure the LLM (gpt2) and the embedding model (all-MiniLM-L6-v2).
    # VectorStoreIndex.from_documents(documents) builds the vector index;
    # index.storage_context.persist saves it into storage.

def build_index():
    """Load documents and build a vector store index."""
    documents = SimpleDirectoryReader(DATA_DIR).load_data()

    # Configure settings
    Settings.llm = HuggingFaceLLM(model_name="gpt2", max_new_tokens=32)
    Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
    return index

In [13]:
# Loading or Creating the Index (load_or_build_index)
    # If storage/ already exists, the index is reloaded from disk using StorageContext.from_defaults.
    # Otherwise, build_index is called to generate a new index.

def load_or_build_index():
    """Load an existing index from disk, otherwise build a new one."""
    if PERSIST_DIR.exists():
        storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
        # Configure settings for loading
        Settings.llm = HuggingFaceLLM(model_name="gpt2", max_new_tokens=32)
        Settings.embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
        return load_index_from_storage(storage_context)
    return build_index()

In [22]:
# Downloading a PDF (download_file)
    # Uses requests to retrieve a PDF from a URL and save it into data/.

import requests

def download_file(url, dest_folder):
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)
    filename = url.split('/')[-1]
    filepath = os.path.join(dest_folder, filename)
    if not os.path.exists(filepath):
        r = requests.get(url, stream=True)
        with open(filepath, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded {filename} to {dest_folder}")
    else:
        print(f"{filename} already exists in {dest_folder}")

In [25]:
# Adding a the fetch_data function,
    # Purpose is to wrap requests.get in a try/except block, display a message in case of a network error, and then re-raise the exception.
    # HTTP status code check
    # ... raising an exception when response.status_code is not 200.

import requests

def fetch_data(url: str) -> str:
    """Fetch content from ``url``.

    The request is wrapped in a ``try``/``except`` block to catch network
    related errors. The HTTP status code is checked and an ``Exception`` is
    raised when a non-success status code is returned.
    """
    try:
        response = requests.get(url, timeout=5)
    except requests.RequestException as exc:
        # Any network related error (DNS failure, refused connection, etc.)
        # is caught here and re-raised after logging/printing.
        print(f"Network error: {exc}")
        raise

    if response.status_code != 200:
        raise Exception(f"Failed to fetch data: status code {response.status_code}")

    return response.text


if __name__ == "__main__":
    # Example usage
    import sys

    url = sys.argv[1] if len(sys.argv) > 1 else "https://example.com"
    try:
        content = fetch_data(url)
        print(content[:200])
    except Exception as err:
        print(err)

Network error: Invalid URL '-f': No scheme supplied. Perhaps you meant https://-f?
Invalid URL '-f': No scheme supplied. Perhaps you meant https://-f?


In [26]:
# Main Block (if __name__ == "__main__":)
    # Downloads a sample PDF (“Attention Is All You Need”).
    # Builds or reloads the index, then runs a query (“What does the quick brown fox do?”).
    # Displays the question and the answer returned by the query engine.

if __name__ == "__main__":
    # Download a sample PDF file
    sample_pdf_url = "https://arxiv.org/pdf/1706.03762.pdf" # Example PDF URL (Attention Is All You Need)
    download_file(sample_pdf_url, DATA_DIR)

    index = load_or_build_index()
    query_engine = index.as_query_engine()
    question = "What does the quick brown fox do?"
    response = query_engine.query(question)
    print(f"Q: {question}\nA: {response}\n")

1706.03762.pdf already exists in data
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/docstore.json.
Loading llama_index.core.storage.kvstore.simple_kvstore from storage/index_store.json.


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Q: What does the quick brown fox do?
A: 3: deadbleity<iquitybleity: deadbleity<iquitybleity: deadbleity<iquitybleity: deadbleity<iquityble

